In [ ]:
# ESTA bake-off, manual run. Before "Run all":
#  1. Settings (right panel, or the three-dot menu on a phone): Accelerator = GPU T4 x1, Internet = on.
#  2. Add Input > Your Work: the notebook "esta-bakeoff-main-kernel", and the datasets
#     "esta-bakeoff-cuts-data" and "esta-bakeoff-frames-data".
#  3. Optional: Add-ons > Secrets, add HF_TOKEN (your Hugging Face token) and tick it for this notebook.
# Then edit the two lines below if you want a different run.

JOBS = ["tags"]              # any of "cuts", "tags", "theme"
MODELS = ["gemma4-e4b"]      # [] = every candidate in those jobs; names: qwen3-vl-8b, qwen3-vl-4b,
                             # gemma3-4b, gemma4-e4b, siglip2, clip-vit-b32, dinov3-small, transnetv2, pyscenedetect


In [ ]:
# Read the answer key (and HF token) back from the previous automated run's notebook.
import ast, glob, json, os

def _find_previous_source():
    for path in glob.glob("/kaggle/input/**/*.ipynb", recursive=True) + glob.glob("/kaggle/input/**/*.py", recursive=True):
        try:
            text = open(path, encoding="utf-8").read()
        except Exception:
            continue
        if path.endswith(".ipynb"):
            try:
                text = "".join("".join(c.get("source", "")) for c in json.loads(text)["cells"] if c.get("cell_type") == "code")
            except Exception:
                continue
        if "GT = json.loads(" in text:
            return path, text
    raise RuntimeError("Attach the notebook esta-bakeoff-main-kernel as an input (Add Input > Your Work > Notebooks).")

_path, _src = _find_previous_source()
_values = {}
for node in ast.walk(ast.parse(_src)):
    if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
        name = node.targets[0].id
        if name == "GT" and isinstance(node.value, ast.Call) and node.value.args:
            _values["GT"] = ast.literal_eval(node.value.args[0])
        elif name == "HF_TOKEN":
            try:
                _values["HF_TOKEN"] = ast.literal_eval(node.value)
            except Exception:
                pass
_GT_JSON = _values["GT"]
_HF = _values.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    _HF = UserSecretsClient().get_secret("HF_TOKEN") or _HF
except Exception:
    pass
_gt = json.loads(_GT_JSON)
_first = next(iter(_gt))
_VIDEO_PROBE = _gt[_first]["video_file"]
_FRAME_PROBE = "%s__shot_%03d.jpg" % (_first, int(_gt[_first]["shots"][0]["shot_id"]))
os.makedirs("/kaggle/working", exist_ok=True)
json.dump(_gt, open("/kaggle/working/gt.json", "w"))
print("answer key from", _path, "-", sum(len(v["shots"]) for v in _gt.values()), "shots;", "HF token set" if _HF else "no HF token")


In [ ]:
# ESTA - M5.1 bake-off: run cut/tag/theme candidates and record their raw
# predictions. Scoring happens locally (tools/match/bakeoff/scoring.py).
import subprocess, sys, json, os, gc, glob, traceback, time

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scenedetect[opencv]", "transnetv2-pytorch",
                "git+https://github.com/huggingface/transformers", "accelerate",
                "bitsandbytes", "einops", "timm"], check=False)

import numpy as np
import torch
import cv2
from PIL import Image

GT = json.loads(_GT_JSON)
JOBS = set(JOBS)
MODELS = set(MODELS)
HF_TOKEN = _HF
OUT = "/kaggle/working"
WORK = "/kaggle/temp"
MAX_SIDE = 448
os.makedirs(WORK, exist_ok=True)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "/kaggle/temp/hf"

def wanted(job, name):
    return job in JOBS and (not MODELS or name in MODELS)

def _resolve_input_dir(expected_slug, must_contain):
    # With 2+ datasets attached, Kaggle mounts them under
    # /kaggle/input/datasets/<owner>/<slug>/ rather than /kaggle/input/<slug>/.
    direct = os.path.join("/kaggle/input", expected_slug)
    if os.path.exists(os.path.join(direct, must_contain)):
        return direct
    hits = glob.glob(os.path.join("/kaggle/input", "**", must_contain), recursive=True)
    return os.path.dirname(hits[0]) if hits else direct

IN_VIDEOS = _resolve_input_dir('esta-bakeoff-cuts-data', _VIDEO_PROBE)
IN_FRAMES = _resolve_input_dir('esta-bakeoff-frames-data', _FRAME_PROBE)

# The T4 has no bf16 hardware: bf16 runs emulated and several times slower.
# fp16 is native, but some models (Gemma in particular) overflow in fp16, so a
# model is loaded fp16 first and reloaded bf16 if its first answers are garbage.
FAST = torch.float16 if torch.cuda.is_available() and not torch.cuda.is_bf16_supported() else torch.bfloat16

results = {"cuts": {}, "tags": {}, "theme": {}, "timing": {}, "errors": {}, "dtype": {}}

def finish():
    with open(os.path.join(OUT, "bakeoff_preds.json"), "w") as fh:
        json.dump(results, fh)
    print("ESTA_BAKEOFF_RESULT::" + json.dumps({"ok": True}))

def purge():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def timed(key, t0):
    dur_min = sum(GT[s]["duration"] for s in GT) / 60
    secs = time.time() - t0
    results["timing"][key] = {"sec_per_min": secs / dur_min,
                              "gpu_min": secs / 60 if torch.cuda.is_available() else 0.0}

def video_path(slug):
    src = os.path.join(IN_VIDEOS, GT[slug]["video_file"])
    dst = os.path.join(WORK, GT[slug]["video_file"])
    if not os.path.exists(dst):
        import shutil
        shutil.copy(src, dst)
    return dst

# ============================== CUTS ==========================================
if wanted("cuts", "pyscenedetect"):
    t0 = time.time()
    try:
        from scenedetect import detect, AdaptiveDetector
        results["cuts"]["pyscenedetect"] = {
            slug: [sc[0].get_seconds() for sc in detect(video_path(slug), AdaptiveDetector())[1:]] for slug in GT}
        timed("cuts:pyscenedetect", t0)
    except Exception as e:
        results["errors"]["cuts:pyscenedetect"] = str(e)[:300]
        traceback.print_exc()
    finish()

if wanted("cuts", "transnetv2"):
    t0 = time.time()
    try:
        import csv
        preds = {}
        for slug in GT:
            out_csv = os.path.join(WORK, slug + "_tn2.csv")
            r = subprocess.run(["transnetv2_pytorch", video_path(slug), "--output", out_csv, "--quiet"],
                               capture_output=True, text=True, timeout=1800)
            if r.returncode != 0 or not os.path.exists(out_csv):
                raise RuntimeError("transnetv2_pytorch failed: " + (r.stderr or r.stdout)[-300:])
            with open(out_csv, newline="") as fh:
                rows = list(csv.DictReader(fh))
            preds[slug] = [round(float(row["end_time"]), 3) for row in rows[:-1]]
        results["cuts"]["transnetv2"] = preds
        timed("cuts:transnetv2", t0)
    except Exception as e:
        results["errors"]["cuts:transnetv2"] = str(e)[:300]
        traceback.print_exc()
    purge()
    finish()

# ============================== SHOTS =========================================
# Shot i spans cuts[i-1]..cuts[i]. Three frames per shot (one keyframe can't
# show whether a shot moves), downscaled so a VLM spends few tokens on each,
# plus a measured motion level the prompt passes on as a hint.
def shot_frames(slug, shot_id):
    cuts = GT[slug]["cuts"]
    i = int(shot_id)
    a, b = cuts[i - 1], cuts[i]
    cap = cv2.VideoCapture(video_path(slug))
    def grab(t):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, img = cap.read()
        return img if ok else None
    frames = []
    for f in (0.2, 0.5, 0.8):
        img = grab(a + (b - a) * f)
        if img is not None:
            h, w = img.shape[:2]
            k = MAX_SIDE / max(h, w)
            frames.append(Image.fromarray(cv2.cvtColor(cv2.resize(img, (int(w * k), int(h * k))), cv2.COLOR_BGR2RGB)))
    grays = []
    for k in range(8):
        img = grab(a + (b - a) * (0.05 + 0.9 * k / 7))
        if img is not None:
            grays.append(cv2.cvtColor(cv2.resize(img, (160, 90)), cv2.COLOR_BGR2GRAY).astype(np.float32))
    cap.release()
    motion = float(np.mean([np.abs(x - y).mean() for x, y in zip(grays, grays[1:])])) if len(grays) > 1 else 0.0
    return frames, motion

SHOTS = []  # (key, frames, motion)
if "tags" in JOBS:
    for slug, v in GT.items():
        for s in v["shots"]:
            try:
                frames, motion = shot_frames(slug, s["shot_id"])
                if frames:
                    SHOTS.append((f"{slug}/{s['shot_id']}", frames, motion))
            except Exception as e:
                results["errors"][f"frames:{slug}/{s['shot_id']}"] = str(e)[:200]
    results["motion"] = {k: m for k, _, m in SHOTS}

PROMPT = (
    "These are 3 frames (start, middle, end) from ONE shot of an edited short-form video. "
    "Measured motion across the shot: {motion}. Reply with ONLY a JSON object:\n"
    '{{"kind": one of ["footage","still","graphic","talking_head","meme"], '
    '"text_on_screen": true/false, "panels": 1, 2 or 3, "overlay": true/false, '
    '"clips_in_shot": integer}}\n'
    "kind: footage = real camera footage, screen recording or game capture; "
    "still = a photo or screenshot shown without its own motion (a slow zoom or pan on it is still a still); "
    "graphic = motion graphic, animated text card, chart or title card; "
    "talking_head = a person speaking to camera; meme = meme template or reaction clip. "
    "text_on_screen = any readable words or numbers on screen, captions included. "
    "panels = how many separate pictures share the frame side by side or stacked (3 = 3 or more). "
    "overlay = a graphic, sticker or text card floating over footage. "
    "clips_in_shot = how many different source clips appear within this one shot."
)

def motion_word(m):
    return "none" if m < 1.0 else "low" if m < 4.0 else "high"

def parse_label_json(text):
    try:
        obj = json.loads(text[text.index("{"): text.rindex("}") + 1])
        return {"kind": str(obj.get("kind", "")).strip().lower(),
                "text_on_screen": bool(obj.get("text_on_screen", False)),
                "panels": int(obj.get("panels", 1)), "overlay": bool(obj.get("overlay", False)),
                "clips_in_shot": int(obj.get("clips_in_shot", 1))}
    except Exception:
        return None

def run_vlm(load, ask):
    # load(dtype) -> (model, processor); ask(model, processor, frames, prompt) -> text.
    preds = {}
    for dtype in (FAST, torch.bfloat16) if FAST != torch.bfloat16 else (torch.bfloat16,):
        model, processor = load(dtype)
        preds = {}
        for n, (key, frames, motion) in enumerate(SHOTS):
            preds[key] = parse_label_json(ask(model, processor, frames, PROMPT.format(motion=motion_word(motion))))
            # fp16 overflow shows up as unparseable output from the start; retry in bf16.
            if n == 3 and dtype != torch.bfloat16 and sum(p is None for p in preds.values()) >= 3:
                break
        else:
            del model
            purge()
            return preds, str(dtype)
        del model
        purge()
    return preds, "bfloat16"

def chat_ask(model, processor, frames, prompt):
    messages = [{"role": "user", "content": [*({"type": "image", "image": f} for f in frames),
                                              {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def qwen_loader(model_id, four_bit=False):
    def load(dtype):
        from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
        kw = dict(dtype=dtype, device_map="auto")
        if four_bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)
        processor = AutoProcessor.from_pretrained(model_id)
        return Qwen3VLForConditionalGeneration.from_pretrained(model_id, **kw), processor
    return load

def auto_loader(model_id, **proc_kw):
    def load(dtype):
        from transformers import AutoModelForImageTextToText, AutoProcessor
        processor = AutoProcessor.from_pretrained(model_id, **proc_kw)
        return AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto"), processor
    return load

TAGGERS = [
    ("qwen3-vl-8b", qwen_loader("Qwen/Qwen3-VL-8B-Instruct", four_bit=True)),
    ("qwen3-vl-4b", qwen_loader("Qwen/Qwen3-VL-4B-Instruct")),
    ("gemma3-4b", auto_loader("google/gemma-3-4b-it")),
    ("gemma4-e4b", auto_loader("google/gemma-4-E4B-it", padding_side="left")),
]
for name, load in TAGGERS:
    if not wanted("tags", name):
        continue
    t0 = time.time()
    try:
        preds, used = run_vlm(load, chat_ask)
        results["tags"][name] = preds
        results["dtype"][name] = used
        timed("tags:" + name, t0)
    except Exception as e:
        results["errors"]["tags:" + name] = str(e)[:300]
        traceback.print_exc()
    purge()
    finish()

# ============================== THEME =========================================
def image_embedding(out):
    # Newer transformers return a ModelOutput from get_image_features; the
    # embedding is its pooler_output (projected), not the per-patch states.
    if isinstance(out, torch.Tensor):
        return out[0]
    pooled = getattr(out, "pooler_output", None)
    return pooled[0] if pooled is not None else out.last_hidden_state[0].mean(dim=0)

def theme_sims(embed):
    pos = [p for p in sorted(glob.glob(os.path.join(IN_FRAMES, "*__shot_*.jpg"))) if not os.path.basename(p).startswith("negative__")]
    neg = sorted(glob.glob(os.path.join(IN_FRAMES, "negative__*.jpg")))
    def norm(v):
        v = v.float().cpu().numpy().reshape(-1)
        return v / (np.linalg.norm(v) + 1e-9)
    pe = np.stack([norm(embed(Image.open(p).convert("RGB"))) for p in pos])
    ne = np.stack([norm(embed(Image.open(p).convert("RGB"))) for p in neg])
    # Leave-one-out centroid, so a frame isn't scored against itself.
    total = pe.sum(axis=0)
    pos_sims = [float(v @ ((total - v) / (np.linalg.norm(total - v) + 1e-9))) for v in pe]
    centroid = total / (np.linalg.norm(total) + 1e-9)
    return {"pos": pos_sims, "neg": [float(v @ centroid) for v in ne]}

def hf_image_embedder(model_cls, proc_cls, model_id, via_features=True):
    import transformers
    processor = getattr(transformers, proc_cls).from_pretrained(model_id)
    model = getattr(transformers, model_cls).from_pretrained(model_id).to(
        "cuda" if torch.cuda.is_available() else "cpu").eval()
    def embed(img):
        inputs = processor(images=[img], return_tensors="pt").to(model.device)
        with torch.no_grad():
            return image_embedding(model.get_image_features(**inputs) if via_features else model(**inputs))
    return model, embed

EMBEDDERS = [
    ("siglip2", ("AutoModel", "AutoProcessor", "google/siglip2-base-patch16-224", True)),
    ("clip-vit-b32", ("CLIPModel", "CLIPProcessor", "openai/clip-vit-base-patch32", True)),
    ("dinov3-small", ("AutoModel", "AutoImageProcessor", "facebook/dinov3-convnext-small-pretrain-lvd1689m", False)),
]
for name, spec in EMBEDDERS:
    if not wanted("theme", name):
        continue
    t0 = time.time()
    try:
        model, embed = hf_image_embedder(*spec)
        results["theme"][name] = theme_sims(embed)
        del model
        timed("theme:" + name, t0)
    except Exception as e:
        results["errors"]["theme:" + name] = str(e)[:300]
        traceback.print_exc()
    purge()
    finish()

finish()
print("ESTA_BAKEOFF_DONE")


In [ ]:
# Scoring (tools/match/bakeoff/scoring.py)
"""Scoring for the M5.1 bake-off. The Kaggle kernel only records raw predictions
(bakeoff_preds.json); every metric is computed here, locally, so a change to how
a job is scored never costs another GPU run."""

from collections import Counter

# The answer key was labelled with seven kinds. No model can tell AI-generated
# footage from real, and screen recordings / AI clips play the same role as
# footage in an edit, so scoring folds them into five classes that map onto the
# plan's shot types.
KIND_MAP = {"footage": "footage", "screen": "footage", "ai": "footage", "still": "still",
            "graphic": "graphic", "talking_head": "talking_head", "meme": "meme"}
KINDS = ["footage", "still", "graphic", "talking_head", "meme"]
TAG_FIELDS = ["kind", "text_on_screen", "panels", "overlay", "clips_in_shot"]


def kind5(k: str) -> str:
    return KIND_MAP.get(str(k).strip().lower(), str(k).strip().lower())


def tag_value(field: str, label: dict):
    v = label.get(field)
    if field == "kind":
        return kind5(v)
    if field == "panels":
        return min(3, max(1, int(v or 1)))
    if field == "clips_in_shot":
        return "1" if int(v or 1) <= 1 else "2+"
    return bool(v)


def f1_cuts(pred: list, gt: list, tol: float = 0.1) -> float:
    used = [False] * len(gt)
    gt_sorted = sorted(gt)
    tp = 0
    for p in sorted(pred):
        best_j, best_d = -1, tol + 1e-9
        for j, g in enumerate(gt_sorted):
            d = abs(p - g)
            if not used[j] and d <= tol and d < best_d:
                best_d, best_j = d, j
        if best_j >= 0:
            used[best_j] = True
            tp += 1
    fp, fn = len(pred) - tp, len(gt) - tp
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def score_field(pairs: list[tuple]) -> dict:
    """pairs: [(predicted, truth)]. Accuracy alone rewards always guessing the
    common answer, so the bar is on balanced accuracy (mean per-class recall),
    reported next to the always-guess-the-majority baseline. A field whose key
    holds a single value can't show a model is any better than that guess, so it
    is marked untestable instead of passing."""
    truths = Counter(t for _, t in pairs)
    n = len(pairs)
    if not n:
        return {"n": 0, "testable": False, "reason": "no predictions"}
    majority, majority_n = truths.most_common(1)[0]
    accuracy = sum(1 for p, t in pairs if p == t) / n
    recalls = {c: sum(1 for p, t in pairs if t == c and p == c) / k for c, k in truths.items()}
    out = {"n": n, "accuracy": accuracy, "baseline": majority_n / n, "baseline_answer": majority,
           "balanced_accuracy": sum(recalls.values()) / len(recalls),
           "per_class_recall": recalls, "support": dict(truths)}
    if len(truths) < 2:
        return {**out, "testable": False, "reason": f"every shot in the key is {majority!r}"}
    return {**out, "testable": True}


def score_tags(preds: dict, gt: dict) -> dict:
    """preds: {"<slug>/<shot_id>": label dict or None}; gt: {slug: {"shots": [...]}}.
    A shot the model gave no parseable answer for counts as wrong, not skipped."""
    fields = {}
    missing = 0
    for field in TAG_FIELDS:
        pairs = []
        for slug, v in gt.items():
            for s in v["shots"]:
                p = preds.get(f"{slug}/{s['shot_id']}")
                if p is None:
                    if field == "kind":
                        missing += 1
                    pairs.append(("<no answer>", tag_value(field, s)))
                else:
                    pairs.append((tag_value(field, p), tag_value(field, s)))
        fields[field] = score_field(pairs)
    return {"fields": fields, "unparsed": missing}


def auc(pos: list[float], neg: list[float]) -> float:
    """Probability a same-style frame scores above an other-style one (ties half)."""
    if not pos or not neg:
        return 0.0
    wins = sum(1.0 if p > q else 0.5 if p == q else 0.0 for p in pos for q in neg)
    return wins / (len(pos) * len(neg))


def label_counts(gt: dict) -> dict:
    return {field: dict(Counter(str(tag_value(field, s)) for v in gt.values() for s in v["shots"]))
            for field in TAG_FIELDS}


In [ ]:
# Scores, as `kaggle.py apply` would compute them.
print()
for name, preds in results["tags"].items():
    out = score_tags(preds, GT)
    print(f"== {name} ({results['dtype'].get(name, '')}, {results['timing'].get('tags:' + name, {}).get('gpu_min', 0):.1f} GPU-min, {out['unparsed']} unparsed)")
    for field, sc in out["fields"].items():
        if sc.get("testable"):
            print(f"  {field:15s} balanced {sc['balanced_accuracy']:.3f}  accuracy {sc['accuracy']:.3f}  always-guess {sc['baseline']:.3f}")
        else:
            print(f"  {field:15s} untestable: {sc.get('reason')}")
for name, sims in results["theme"].items():
    print(f"== {name}: AUC {auc(sims['pos'], sims['neg']):.3f}")
for name, by_slug in results["cuts"].items():
    gt_cuts = {s: v["cuts"][1:-1] for s, v in GT.items()}
    w = {s: len(gt_cuts[s]) for s in GT}
    print(f"== {name}: F1 {sum(f1_cuts(by_slug.get(s, []), gt_cuts[s]) * n for s, n in w.items()) / (sum(w.values()) or 1):.3f}")
if results["errors"]:
    print("errors:", json.dumps(results["errors"], indent=1))
print("Files to keep: /kaggle/working/bakeoff_preds.json and /kaggle/working/gt.json (Output tab).")
